# Preprocessing: Customer Churn Prediction

This notebook prepares the raw dataset for modeling. The following sections apply each preprocessing step in turn: correcting data types, removing unique identifiers, encoding categorical features, splitting the data into training and test sets, and scaling numeric columns for use in machine learning models.

## What this pipeline does

1. **Load** the CSV into a DataFrame.
2. **Repair `TotalCharges`**: values are in string format; new customers have blanks that become missing after conversion.
3. **Drop `customerID`**: unique per row, not a feature for prediction.
4. **Binary encoding**: map Yes/No columns (including the target `Churn`) to 1/0.
5. **One-hot encoding**: turn remaining categorical columns into numeric dummy columns.
6. **Train–test split**: reserve 20% for evaluation, with stratification so churn rate is similar in train and test.
7. **Standard scaling**: scale `tenure`, `MonthlyCharges`, and `TotalCharges` using statistics learned **only from the training split** 

## Imports

- **`_setup`**: adds the project root to `sys.path` so packages under `src/` resolve when the notebook runs from `notebooks/`.
- **`DATA_PATH`** (`from src.config`): absolute path to the churn CSV (`data/customer_churn.csv` relative to the project root).
- **pandas / NumPy**: table operations and arrays
- **StandardScaler**: for scaling numerical columns
- **train_test_split**: for splitting the dataset into train and test sets

In [53]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import _setup
from src.config import DATA_PATH

print('loaded all dependencies successfully...')

loaded all dependencies successfully...


## Load the dataset

The CSV is read with **`pd.read_csv(DATA_PATH)`**, using the path from `src.config`

In [54]:
df = pd.read_csv(DATA_PATH)

print('dataset loaded successfully...')

dataset loaded successfully...


## `TotalCharges`: string to number and missing values

`TotalCharges` is stored as text. `pd.to_numeric(df['TotalCharges'], errors='coerce')` turns invalid or empty strings into NaN. Rows that had no total charge yet (common for brand-new accounts) are filled with `0` 

In [55]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

## Drop `customerID`

The customer identifier is unique per row and does not generalize; keeping it would let a model memorize rows instead of learning patterns.

In [56]:
df.drop('customerID', axis=1, inplace=True)

## Encode the target: `Churn`

Supervised learning needs a numeric label. We map **Yes → 1** and **No → 0** so churn is the positive class.

In [57]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## Encode other Yes/No columns

`Partner`, `Dependents`, `PhoneService`, and `PaperlessBilling` use the same Yes/No scheme. `SeniorCitizen` is already 0/1 in this dataset, so it is not listed here.

In [58]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

## One-hot encoding (`pd.get_dummies`)

- **`drop_first=True`**: drops the first category per feature to reduce redundancy (avoids perfect multicollinearity between dummies).
- **`dtype=int`**: dummy columns are 0/1 integers instead of bool

All non-numeric columns still present are converted; the result is a fully numeric matrix suitable for many sklearn estimators.

In [59]:
df = pd.get_dummies(df, drop_first=True, dtype=int)

## Features, labels, and train–test split

- **`X`** : all columns except `Churn`.
- **`y`** : Only `Churn`.
- **`test_size=0.2`** : 80% train, 20% test.
- **`stratify=y`** : preserves the approximate proportion of churners in both splits (helpful as the positive class is rare).
- **`random_state=42`** : makes the split reproducible

`train_test_split` returns **`X_train`, `X_test`, `y_train`, `y_test`** in that order.

In [60]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

## Standardize selected numeric columns

We scale **`tenure`**, **`MonthlyCharges`**, and **`TotalCharges`** 

**Important:** Called **`fit_transform` only on `X_train`**, then **`transform` on `X_test`** with the same scaler. Fitting on the full dataset (or on the test fold) leaks information from the test distribution into training.

In [61]:
scaler = StandardScaler()

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])